In [1]:
from discovery_child_development.getters.openalex import get_sentence_embeddings
from discovery_child_development import PROJECT_DIR, logging, config, S3_BUCKET
import pandas as pd
from discovery_child_development.utils.jsonl_utils import load_jsonl

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

# Path to sentence embeddings
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE = "sentence_vectors_384_labelled.parquet"

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"
# Path automatically labelled data folder
LABELS_DIR = PROJECT_DIR / 'outputs/labels/taxonomy_cat'

# Path to models
from discovery_child_development.pipeline.models.taxonomy_cat.train_classifiers import (
    MODEL_PATH,
    S3_MODEL_PATH,
    MODELS_SIMPLE,
)

topic = "mobile"



2024-03-19 15:40:26,170 - botocore.credentials - INFO - Found credentials in environment variables.
2024-03-19 15:40:27,477 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load vectors
embeddings_all = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET, filepath=VECTORS_PATH, filename=VECTORS_FILE, id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
    .set_index("id")
)

# Load automatically labelled data
labels_df = (
    pd.DataFrame(load_jsonl(LABELS_DIR / f"taxonomy_cat_{topic}.jsonl"))
    .assign(id = lambda df: df['id'].apply(lambda x: x.split('/')[-1]))
    .query("id not in @eval_df.id")
    .rename(columns={"prediction": "labels"})
)[['id', 'labels', 'text']]    

# Load model inference
inference_df = pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/taxonomy_cat_predictions_{topic}.csv')


# Reduce dimensionality and plot